# KakaoLens: Klasifikasi Tingkat Fermentasi Biji Kakao

Notebook ini melatih pengklasifikasi per biji yang dipakai backend KakaoLens.
Model: YOLO11n-cls, dilatih di Google Colab dengan GPU T4.

Notebook ini melatih dan membandingkan dua model dengan arsitektur dan data sumber yang
sama, namun memiliki perbedaan pada komposisi kelas di data latih:

- Model **3:1**: seluruh data latih dipakai apa adanya, 140 `fermented` dan 420
  `poorly_fermented`.
- Model **Seimbang (bal)**: `poorly_fermented` di data latih dikurangi supaya jumlahnya sama
  dengan `fermented`, 140 untuk `fermented` dan 140 untuk `under_fermented`.

Test setnya sama untuk keduanya, jadi hasilnya bisa dibandingkan langsung. Model seimbang (bal)
yang akhirnya dipakai di backend (`weights/best.pt`). Model 3:1 hanya dilaporkan sebagai
pembanding.

## Sumber dataset

Mendeley Data `pcx7mj68yn`, Santos dkk., Universidade Estadual de Santa Cruz, Brasil.
Repositori tersebut berisi enam dataset citra turunan. Kami menggunakan varian
`framed_and_centralized`, versi 1 metode A.

Dataset asli memiliki 14 kelas, masing masing 100 gambar. Delapan folder dipakai
(800 gambar), enam sisanya dikeluarkan dengan alasan yang dijelaskan di bagian merge.

## Alur notebook

1. Mount Drive, ekstrak dataset ke disk lokal runtime
2. Merge 14 kelas menjadi 2 kelas
3. Latih model rasio 3:1, evaluasi di test set
4. Latih model komposisi seimbang 1:1, evaluasi di test set yang sama
5. Simpan bobot ke Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
!ls /content/drive/MyDrive/cocoa/cocoa8/

Compartmentalized_Brown		   Compartmentalized_Slaty  Plated_Purple
Compartmentalized_PartiallyPurple  Plated_Brown		    Plated_Slaty
Compartmentalized_Purple	   Plated_PartiallyPurple


In [13]:
!rm -rf /content/raw
!unzip -q /content/drive/MyDrive/cocoa8.zip -d /content/raw

In [15]:
!for d in /content/raw/*/; do echo -n "$(basename "$d"): "; ls "$d" | wc -l; done

Compartmentalized_Brown: 100
Compartmentalized_PartiallyPurple: 100
Compartmentalized_Purple: 100
Compartmentalized_Slaty: 100
Plated_Brown: 100
Plated_PartiallyPurple: 100
Plated_Purple: 100
Plated_Slaty: 100


## Merge 14 kelas jadi 2 kelas

| Kelas keluaran | Folder sumber | Jumlah |
|---|---|---|
| `fermented` | Compartmentalized_Brown, Plated_Brown | 200 |
| `poorly_fermented` | PartiallyPurple, Purple, Slaty (Compartmentalized + Plated) | 600 |

### Folder yang tidak dipakai

**White (2 folder).** Biji putih bukan gradasi tingkat fermentasi. Empat kelas lainnya
berada pada satu sumbu, dari terfermentasi penuh sampai tidak terfermentasi. White di luar
sumbu itu. Kalau dimasukkan ke `poorly_fermented`, angka yang keluar tidak lagi murni
menggambarkan tingkat fermentasi.

**Moldered, Flattened, Brittle, Agglutinated (4 folder).** Cacat fisik dan sanitasi itu
penilaian terpisah dari tingkat fermentasi, dan UI menyatakan parameter itu tidak dinilai.
Kalau dilatih jadi `poorly_fermented`, model diam-diam menilai hal yang UI bilang tidak
dinilai.

### Catatan


Urutan key `MERGE` tidak diubah dari skema 4 kelas, jadi splitnya deterministik dan
proporsi `fermented` tetap 25% (200 gambar).

In [16]:
import random, shutil
from pathlib import Path

SRC = Path("/content/raw")
DST = Path("/content/cocoa5_binary")
SEED = 42

MERGE = {
    "fermented":       ["Compartmentalized_Brown",          "Plated_Brown"],
    "poorly_fermented":["Compartmentalized_PartiallyPurple","Plated_PartiallyPurple",
                        "Compartmentalized_Purple",         "Plated_Purple",
                        "Compartmentalized_Slaty",          "Plated_Slaty"],
}

if DST.exists():
    shutil.rmtree(DST)

rng = random.Random(SEED)
counts = {s: {c: 0 for c in MERGE} for s in ("train", "val", "test")}

for cls, folders in MERGE.items():
    for folder in folders:
        d = SRC / folder
        assert d.is_dir(), f"missing folder: {d}"
        files = sorted(p for p in d.iterdir() if p.is_file())
        assert len(files) == 100, f"{folder} has {len(files)} files, expected 100"

        rng.shuffle(files)
        splits = {"train": files[:70], "val": files[70:85], "test": files[85:]}

        for split, items in splits.items():
            out = DST / split / cls
            out.mkdir(parents=True, exist_ok=True)
            for p in items:
                shutil.copy2(p, out / f"{folder}__{p.name}")
            counts[split][cls] += len(items)

total = 0
for split in ("train", "val", "test"):
    print(split)
    for cls in MERGE:
        n = len(list((DST / split / cls).iterdir()))
        total += n
        print(f"  {cls:16s} {n}")
assert total == 800, f"total is {total}, expected 800"
print("\nOK, total:", total)

train
  fermented        140
  poorly_fermented 420
val
  fermented        30
  poorly_fermented 90
test
  fermented        30
  poorly_fermented 90

OK, total: 800


## Training 1, (komposisi gambar 3:1)

Semua data latih dipakai: 140 `fermented` lawan 420 `poorly_fermented`.
Konfigurasi: `epochs=50 imgsz=224 batch=64 patience=15 hsv_h=0.0 auto_augment=None`.

`hsv_h=0.0` dan `auto_augment=None` supaya augmentasi tidak mengubah warna. Warna adalah
pembeda utama tingkat fermentasi, jadi augmentasi hue akan merusak fitur yang mau dipelajari.

Test set 120 gambar (30 `fermented`, 90 `poorly_fermented`), sama untuk semua model di
notebook ini.

In [ ]:
!pip -q install ultralytics
!yolo classify train model=yolo11n-cls.pt data=/content/cocoa5_binary \
  epochs=50 imgsz=224 batch=64 patience=15 \
  hsv_h=0.0 auto_augment=None project=/content/runs name=cocoa5_binary_v1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cocoa5_binary, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.

In [17]:
!pip -q install ultralytics
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

m = YOLO("/content/drive/MyDrive/cocoa/runs/cocoa5_binary_v1/weights/best.pt")
names = m.names
cm = Counter()

root = Path("/content/cocoa5_binary/test")

for cls_dir in sorted(root.iterdir()):
    true = cls_dir.name
    for p in sorted(cls_dir.iterdir()):
        r = m.predict(p, imgsz=224, verbose=False)[0]
        cm[(true, names[int(r.probs.top1)])] += 1

labels = sorted(names.values())
lw = max(len(l) for l in labels) + 6
cw = max(len(l) for l in labels) + 8

print(" " * lw + "".join(f"{'pred ' + l:>{cw}}" for l in labels))
for t in labels:
    print(f"{'true ' + t:<{lw}}" + "".join(f"{cm[(t, p)]:>{cw}}" for p in labels))

total = sum(cm.values())
correct = sum(cm[(l, l)] for l in labels)
print(f"\naccuracy {correct}/{total} = {correct / total:.3f}\n")

for l in labels:
    tp = cm[(l, l)]
    pl = sum(cm[(t, l)] for t in labels)
    tl = sum(cm[(l, p)] for p in labels)
    print(f"{l:<{lw}}precision {tp:>3}/{pl:<3} = {tp / pl if pl else 0:.3f}"
          f"   recall {tp:>3}/{tl:<3} = {tp / tl if tl else 0:.3f}")

for p in sorted((root / "fermented").iterdir()):
    r = m.predict(p, imgsz=224, verbose=False)[0]
    print(f"{p.name[:40]:42} P(fermented)={float(r.probs.data[0]):.3f}")

                                pred fermented   pred poorly_fermented
true fermented                              20                      10
true poorly_fermented                        4                      86

accuracy 106/120 = 0.883

fermented             precision  20/24  = 0.833   recall  20/30  = 0.667
poorly_fermented      precision  86/96  = 0.896   recall  86/90  = 0.956
Compartmentalized_Brown__CB_004.JPG_p4_v   P(fermented)=1.000
Compartmentalized_Brown__CB_005.JPG_p4_v   P(fermented)=0.999
Compartmentalized_Brown__CB_012.JPG_p4_v   P(fermented)=1.000
Compartmentalized_Brown__CB_014.JPG_p4_v   P(fermented)=0.999
Compartmentalized_Brown__CB_015.JPG_p4_v   P(fermented)=1.000
Compartmentalized_Brown__CB_018.JPG_p4_v   P(fermented)=1.000
Compartmentalized_Brown__CB_029.JPG_p4_v   P(fermented)=0.995
Compartmentalized_Brown__CB_032.JPG_p4_v   P(fermented)=0.999
Compartmentalized_Brown__CB_036.JPG_p4_v   P(fermented)=0.999
Compartmentalized_Brown__CB_055.JPG_p4_v   P(fermented)=

### Temuan Plated_Brown

Sepuluh dari sepuluh kesalahan di kelas `fermented` berasal dari Plated_Brown.
Compartmentalized_Brown benar semua (15/15) dengan P(fermented) >= 0.993. Lima dari sepuluh
kesalahan Plated punya P(fermented) < 0.05, jadi model salah dengan yakin, bukan ragu. Namun ketimpangan ini diselesaikan oleh Model komposisi seimbang(1:1) dibawah.

In [18]:
old = YOLO("/content/drive/MyDrive/cocoa/runs/cocoa4_v1/weights/best.pt")

for p in sorted((root / "fermented").iterdir()):
    r = old.predict(p, imgsz=224, verbose=False)[0]
    pred = old.names[int(r.probs.top1)]
    if pred != "fermented":
        print(f"{p.name[:38]:40} -> {pred}")

Compartmentalized_Brown__CB_087.JPG_p4   -> under_fermented
Plated_Brown__PB_059.JPG_p4_v1.JPG       -> under_fermented
Plated_Brown__PB_064.JPG_p4_v1.JPG       -> under_fermented
Plated_Brown__PB_073.JPG_p4_v1.JPG       -> under_fermented


In [ ]:
!cp -r /content/runs/cocoa5_binary_v1 /content/drive/MyDrive/cocoa/runs/

## Training 2, komposisi seimbang (1:1)

`poorly_fermented` di train dikurangi jadi 140, kuota [47, 47, 46] untuk PartiallyPurple,
Purple, dan Slaty, seed 42. `val` dan `test` tidak disentuh, jadi hasilnya bisa dibandingkan
langsung dengan pelatihan 1.

Terdapat kuota per folder, bukan ambil acak dari gabungan tiga folder, agar ketiga sumber tetap
terwakili jumlahnya dengan seimbang.

In [ ]:
import random, shutil
from pathlib import Path

SRC = Path("/content/cocoa5_binary")
DST = Path("/content/cocoa5_binary_bal")
SEED = 42

if DST.exists():
    shutil.rmtree(DST)

for split in ("train", "val", "test"):
    for cls_dir in sorted((SRC / split).iterdir()):
        out = DST / split / cls_dir.name
        out.mkdir(parents=True, exist_ok=True)
        for p in sorted(cls_dir.iterdir()):
            shutil.copy2(p, out / p.name)

# train only: cut poorly_fermented down to 140, evenly across its three source folders
pf = DST / "train" / "poorly_fermented"
files = sorted(pf.iterdir())
groups = {}
for p in files:
    groups.setdefault(p.name.split("__")[0].replace("Compartmentalized_", "").replace("Plated_", ""), []).append(p)

rng = random.Random(SEED)
keep = set()
quota = [47, 47, 46]
for (g, items), q in zip(sorted(groups.items()), quota):
    rng.shuffle(items)
    keep.update(items[:q])
    print(f"{g:20} {len(items)} -> {q}")

for p in files:
    if p not in keep:
        p.unlink()

for split in ("train", "val", "test"):
    for cls_dir in sorted((DST / split).iterdir()):
        print(split, cls_dir.name, len(list(cls_dir.iterdir())))

PartiallyPurple      140 -> 47
Purple               140 -> 47
Slaty                140 -> 46
train fermented 140
train poorly_fermented 140
val fermented 30
val poorly_fermented 90
test fermented 30
test poorly_fermented 90


In [ ]:
!yolo classify train model=yolo11n-cls.pt data=/content/cocoa5_binary_bal \
  epochs=50 imgsz=224 batch=64 patience=15 \
  hsv_h=0.0 auto_augment=None project=/content/runs name=cocoa5_binary_bal_v1

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cocoa5_binary_bal, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cocoa5_binary_bal_v1, nbs=64, nms=False,

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

m = YOLO("/content/drive/MyDrive/cocoa/runs/cocoa5_binary_bal_v1/weights/best.pt")
names = m.names

root = Path("/content/cocoa5_binary_bal/test")
cm = Counter()

for cls_dir in sorted(root.iterdir()):
    true = cls_dir.name
    for p in sorted(cls_dir.iterdir()):
        r = m.predict(p, imgsz=224, verbose=False)[0]
        cm[(true, names[int(r.probs.top1)])] += 1

labels = sorted(names.values())
lw = max(len(l) for l in labels) + 6
cw = max(len(l) for l in labels) + 8

print(" " * lw + "".join(f"{'pred ' + l:>{cw}}" for l in labels))
for t in labels:
    print(f"{'true ' + t:<{lw}}" + "".join(f"{cm[(t, p)]:>{cw}}" for p in labels))

total = sum(cm.values())
correct = sum(cm[(l, l)] for l in labels)
print(f"\naccuracy {correct}/{total} = {correct / total:.3f}\n")

for l in labels:
    tp = cm[(l, l)]
    pl = sum(cm[(t, l)] for t in labels)
    tl = sum(cm[(l, p)] for p in labels)
    print(f"{l:<{lw}}precision {tp:>3}/{pl:<3} = {tp / pl if pl else 0:.3f}"
          f"   recall {tp:>3}/{tl:<3} = {tp / tl if tl else 0:.3f}")

for p in sorted((root / "fermented").iterdir()):
    r = m.predict(p, imgsz=224, verbose=False)[0]
    print(f"{p.name[:40]:42} P(fermented)={float(r.probs.data[0]):.3f}")

                                pred fermented   pred poorly_fermented
true fermented                              27                       3
true poorly_fermented                        7                      83

accuracy 110/120 = 0.917

fermented             precision  27/34  = 0.794   recall  27/30  = 0.900
poorly_fermented      precision  83/86  = 0.965   recall  83/90  = 0.922
Compartmentalized_Brown__CB_004.JPG_p4_v   P(fermented)=0.991
Compartmentalized_Brown__CB_005.JPG_p4_v   P(fermented)=0.591
Compartmentalized_Brown__CB_012.JPG_p4_v   P(fermented)=1.000
Compartmentalized_Brown__CB_014.JPG_p4_v   P(fermented)=0.991
Compartmentalized_Brown__CB_015.JPG_p4_v   P(fermented)=0.987
Compartmentalized_Brown__CB_018.JPG_p4_v   P(fermented)=1.000
Compartmentalized_Brown__CB_029.JPG_p4_v   P(fermented)=0.856
Compartmentalized_Brown__CB_032.JPG_p4_v   P(fermented)=0.932
Compartmentalized_Brown__CB_036.JPG_p4_v   P(fermented)=0.943
Compartmentalized_Brown__CB_055.JPG_p4_v   P(fermented)=

### Hasil pelatihan 2

Kesalahan sistematis di Plated_Brown hilang setelah komposisi latih diseimbangkan.

Bias agregat di test set yang sama (proporsi sebenarnya 25% fermented): model 3:1 melaporkan 20.0%, model seimbang 28.3%. Bias model seimbang lebih kecil karena kesalahannya ke dua arah (FN dan FP) dan sebagian saling menghapus.

### Batasan

1. **Dua hal berubah bersamaan.** Dari 3:1 ke 1:1, rasio kelas berubah dan jumlah data latih
   turun dari 560 ke 280. Efek penyeimbangan kelas tidak bisa diklaim sendirian dikarenakan kita tidak dapat mengetahui, apakah masalah ini diselesaikan oleh rasio yang seimbang atau jumlah `poorly_fermented` yang berkurang.

2. **Test set kecil di kelas minoritas.** Hanya 30 biji `fermented`, maka selisih 110 dan 106 hanya 4 biji. Yang dapat diklaim bukan "lebih akurat", namun "kesalahannya ke dua arah
   (7 dan 3), bukan menumpuk di satu sisi (4 dan 10)".

3. **Semua angka ini dari kondisi foto yang sama dengan data latih.** Model yang bagus di kondisi ini belum tentu bertahan di foto ponsel dan biji kakao Indonesia. Angka Santos dipakai untuk memilih kandidat
   model, evaluasi di biji lokal terpisah dan ada di proposal.

In [ ]:
!cp -r /content/runs/cocoa5_binary_bal_v1 /content/drive/MyDrive/cocoa/runs/

### Kesimpulan

Model komposisi seimbang (1:1) adalah model yang dipakai di backend. Dasar pemilihannya:

- Recall `fermented` 27/30 lawan 20/30. Kelas ini yang paling penting karena
  proporsi biji terfermentasi yang baik adalah output utama produk.
- Kesalahannya tersebar ke dua arah, bukan menumpuk di satu sisi. Akibatnya bias
  persentase agregat lebih kecil, dan produk melaporkan persentase.
- Kegagalan sistematis di Plated_Brown tidak muncul lagi.

Pemilihan ini dilakukan di test set Santos yang terkontrol, bukan di foto biji lokal.
Verifikasi bahwa model masih bekerja pada biji Indonesia dilakukan terpisah dengan
`gate_labels.csv`, dan hasilnya dilaporkan di proposal.